# 03 — Baselines and the cost modelTwo things that decide whether any result here means anything: what "doing nothingclever" scores, and how much trading actually costs.

In [ ]:
import sys, warningssys.path.insert(0, "../src")warnings.filterwarnings("ignore")import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom stock_movement.config import load_configfrom stock_movement.dataset import build_datasetconfig = load_config("../configs/reproduction/readme_aapl_2026_07.yaml")dataset = build_dataset(config)dataset.summary()

## Baselines on development folds onlyThese are scored through the identical protocol as every real candidate, on theidentical folds — never on the holdout.

In [ ]:
from stock_movement.selection import baseline_candidates, evaluate_candidate, development_boundsfrom stock_movement.split import walk_forward_foldsn_dev, test_start = development_bounds(dataset, config)folds = walk_forward_folds(n_dev, config.split.walk_forward_splits, gap=config.split.gap)print(f"development: {dataset.index[0].date()} .. {dataset.index[n_dev - 1].date()} ({n_dev} rows)")print(f"holdout sealed from {dataset.index[test_start].date()}")results = [evaluate_candidate(spec, dataset, folds, config) for spec in baseline_candidates(config)]pd.DataFrame([r.summary_row() for r in results]).set_index("candidate")[    ["balanced_accuracy_mean", "balanced_accuracy_std", "accuracy_mean", "roc_auc_mean", "log_loss_mean"]].round(4)

## The accuracy trap`majority` and `always_up` post the highest *plain* accuracy of anything in thisproject — by predicting the same thing every session. Their balanced accuracy isexactly 0.500. If a report quotes accuracy alone here, distrust it first.

In [ ]:
frame = pd.DataFrame([r.summary_row() for r in results]).set_index("candidate")fig, ax = plt.subplots(figsize=(9, 4.5))frame[["accuracy_mean", "balanced_accuracy_mean"]].plot(kind="bar", ax=ax)ax.axhline(0.5, color="crimson", ls="--", label="coin flip")ax.set_ylim(0.4, 0.6)ax.legend()ax.set_title("Plain accuracy flatters the trivial baselines; balanced accuracy does not")plt.tight_layout()

## The cost model is the dominant termUnder `next_open` execution every active session is a full round trip. This is nota detail — it is the difference between a rising market and a losing strategy.

In [ ]:
from stock_movement.backtest import compute_close_to_close_costs, compute_intraday_round_trip_costssessions = 819always_invested = pd.Series(1.0, index=pd.RangeIndex(sessions))intraday = compute_intraday_round_trip_costs(always_invested, 0.0010).sum()holding = compute_close_to_close_costs(always_invested, 0.0005).sum()print(f"always active, {sessions} sessions")print(f"  next_open (round trip each session): {intraday:.4f}  ({intraday:.1%} of capital)")print(f"  close_to_close (one entry):          {holding:.4f}  ({holding:.2%} of capital)")print(f"  ratio: {intraday / holding:.0f}x")print()print("Charging on position change in intraday mode — the original bug — understated")print("cost by that factor and turned a -8.9% strategy into a reported +81%.")

## What the cost drag does to an always-active strategy

In [ ]:
from stock_movement.backtest import always_active, buy_and_hold_close_to_closefuture = dataset.future_return.iloc[test_start:]prices_test = dataset.prices["Close"].astype(float)intraday_result = always_active(future, config.backtest)hold_result = buy_and_hold_close_to_close(prices_test, config.backtest, index=future.index)fig, ax = plt.subplots(figsize=(11, 5))ax.plot(intraday_result.equity_curve, label=f"always_long_intraday ({intraday_result.metrics['cumulative_return']:+.1%})", lw=2)ax.plot(hold_result.equity_curve, label=f"buy_and_hold ({hold_result.metrics['cumulative_return']:+.1%})", lw=2, ls="--", color="black")ax.set_yscale("log")ax.set_title("Same asset, same window — the only difference is trading frequency")ax.legend()ax.grid(alpha=0.3, which="both")plt.tight_layout()